In [ ]:
print ("Copper Prices Mechanism using Ornstein-Uhlenback Mean Reversion, Markov Regime Change, and Monte-Carlo Simulation")
# COPPER STOCHASTIC PRICING MODEL — CLEAN FINAL VERSION

In [ ]:
!pip install pandas numpy statsmodels matplotlib scipy openpyxl seaborn

In [ ]:
##CELL 2: Imports & Config
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
import scipy.stats as stats
import json, os, warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.family"] = "sans-serif"
 
COPPER_FILE = r"C:\Users\kshit\Documents\Copper prices mechanism\Copper_Prices.xlsx"  ## "copper_prices.csv" if you saved as CSV
DATE_COL    = "Time"                 # exact column name for dates in your file
PRICE_COL   = "CopperPrices"        # exact column name for prices in your file
 
# Model parameters
CALIB_END   = "2019-12-01"          # pre-energy-transition calibration cutoff
BREAK_DATE  = "2020-06-01"          # Chow test break point
N_SIMS      = 10_000                # Monte Carlo simulations per scenario
FC_MONTHS   = 120                   # forecast horizon: 120 months = 10 years to 2036
# Scenario long-run means (USD/tonne) — justify in article
MU_A = 7_500    # ~$3.40/lb  Bear: revert toward historical mean
MU_B = 9_500    # ~$4.31/lb  Base: partial regime shift
MU_C = 11_500   # ~$5.22/lb  Bull: permanent high-demand regime
# Note: $4.80/lb (USGS 2025 projected avg) = $10,584/t — sits between B and C
 
# Colours
CR, CB, CG, CA = "#B5451B", "#2E5FA3", "#27AE60", "#E8A838"
 
print("✅  Config loaded.")

In [ ]:
#CELL 3: Load Copper Price Data 
def load_copper(filepath, date_col, price_col):
    ext = os.path.splitext(filepath)[1].lower()
    if ext in (".xlsx", ".xls"):
        df = pd.read_excel(filepath)
    else:
        df = pd.read_csv(filepath)

    print(f"Columns found: {list(df.columns)}")
    
    # Extract price column
    prices = pd.to_numeric(df[price_col], errors="coerce").dropna().values

    # Generate monthly date index — 410 months from Jan 1992
    dates = pd.date_range(start="1992-01-01", periods=len(prices), freq="MS")

    df_clean = pd.DataFrame({"price": prices}, index=dates)
    df_clean.index.name = "date"
    df_clean = df_clean[df_clean["price"] > 0]

    print(f"\n✅  {len(df_clean)} monthly observations")
    print(f"    {df_clean.index[0].strftime('%Y-%m')}  →  {df_clean.index[-1].strftime('%Y-%m')}")
    print(f"    Range: ${df_clean['price'].min():,.0f} – ${df_clean['price'].max():,.0f} /tonne")
    print(f"    Latest: ${df_clean['price'].iloc[-1]:,.0f} /tonne")
    return df_clean

df = load_copper(COPPER_FILE, DATE_COL, PRICE_COL)
df["log_price"] = np.log(df["price"])
df.head(3)


In [ ]:
# CELL 4: Raw Price History Chart 

# First, check what your dates actually look like
print("Date index sample:")
print(df.index[:5])
print(f"\nFrequency detected: {pd.infer_freq(df.index)}")
print(f"Total observations: {len(df)}")

# If dates look annual (e.g. 1992, 1993...) we need to re-examine the source
# If monthly, the chart fix below will render correctly

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import numpy as np

price = df["price"]

fig, ax = plt.subplots(figsize=(16, 7))
fig.patch.set_facecolor("#FAFAFA")
ax.set_facecolor("#FAFAFA")

# ── Area fill beneath the line (gradient effect via stack) ──────
ax.fill_between(price.index, price.values, price.min() * 0.85,
                alpha=0.08, color="#B5451B")

# ── Main price line ─────────────────────────────────────────────
ax.plot(price.index, price.values,
        color="#1C1C1C", lw=1.6, zorder=4, label="_nolegend_")

# ── Energy transition shading ───────────────────────────────────
et_start = pd.Timestamp("2020-01-01")
ax.axvspan(et_start, price.index[-1],
           alpha=0.06, color="#E8A838", zorder=1)
ax.axvline(et_start, color="#E8A838", lw=1.2, ls="--", alpha=0.7, zorder=3)

# ── Key event annotations ───────────────────────────────────────
events = {
    "2008-09": ("Global\nFinancial Crisis", -1_800, "right"),
    "2011-02": ("China\nSupercycle Peak",    1_200, "center"),
    "2016-01": ("Post-Cycle\nLow",          -1_500, "center"),
    "2020-03": ("COVID-19",                 -1_200, "right"),
    "2025-01": ("Tariff\nShock",             1_200, "left"),
}

for date_str, (label, y_offset, ha) in events.items():
    dt = pd.Timestamp(date_str)
    if price.index[0] <= dt <= price.index[-1]:
        val = price.asof(dt)
        ax.annotate(
            label,
            xy=(dt, val),
            xytext=(0, y_offset),
            textcoords="offset points",
            fontsize=7.8,
            color="#555555",
            ha=ha,
            va="center",
            arrowprops=dict(
                arrowstyle="-",
                color="#AAAAAA",
                lw=0.8,
                connectionstyle="arc3,rad=0.0"
            )
        )

# ── Energy transition label ─────────────────────────────────────
ax.text(pd.Timestamp("2021-06"), price.max() * 0.88,
        "Energy Transition Era",
        fontsize=9, color="#C4882A", fontstyle="italic", fontweight="bold")

# ── Horizontal reference: USGS 2025 projected avg ───────────────
usgs_2025 = 10_582   # $4.80/lb × 2204.6
ax.axhline(usgs_2025, color="#2E5FA3", lw=1, ls=":",
           alpha=0.6, zorder=2)
ax.text(price.index[2], usgs_2025 + 150,
        f"USGS 2025 projected avg: ${usgs_2025:,}/t ($4.80/lb)",
        fontsize=7.5, color="#2E5FA3", alpha=0.85)

# ── Axis formatting ─────────────────────────────────────────────
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.set_ylabel("USD per Metric Tonne", fontsize=11, color="#333333", labelpad=10)
ax.set_xlim(price.index[0], price.index[-1])
ax.set_ylim(price.min() * 0.82, price.max() * 1.08)

# Clean spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color("#DDDDDD")
ax.spines["bottom"].set_color("#DDDDDD")
ax.tick_params(colors="#555555", labelsize=9.5)
ax.grid(axis="y", color="#E5E5E5", lw=0.8, zorder=0)
ax.grid(axis="x", color="#EEEEEE", lw=0.5, zorder=0)

# ── Title block ─────────────────────────────────────────────────
ax.set_title(
    "LME Copper Spot Price  |  1992 – 2026",
    fontsize=15, fontweight="bold", color="#1C1C1C",
    pad=16, loc="left"
)
ax.text(0.0, 1.03,
        "Monthly USD per metric tonne  ·  Source: World Bank Commodity Markets",
        transform=ax.transAxes,
        fontsize=8.5, color="#888888")

# ── Latest price callout box ─────────────────────────────────────
latest_val  = price.iloc[-1]
latest_date = price.index[-1]
ax.annotate(
    f"Latest\n${latest_val:,.0f}/t",
    xy=(latest_date, latest_val),
    xytext=(-55, -40),
    textcoords="offset points",
    fontsize=8.5,
    fontweight="bold",
    color="white",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#1C1C1C", alpha=0.85),
    arrowprops=dict(arrowstyle="->", color="#1C1C1C", lw=1.2)
)

plt.tight_layout()
plt.savefig("chart_00_raw_prices.png", dpi=180, bbox_inches="tight",
            facecolor="#FAFAFA")
plt.show()
print("📊  Saved: chart_00_raw_prices.png")

# ── Diagnose the staircase issue ─────────────────────────────────
print(f"\nIf the chart still looks like a staircase (annual steps),")
print(f"your Time column is being parsed as years, not months.")
print(f"Run this to check:")
print(f"  df.index[:12]")
print(f"Monthly data should show: 1992-01-01, 1992-02-01, 1992-03-01 ...")
print(f"Annual data shows: 1992-01-01, 1993-01-01, 1994-01-01 ...")

In [ ]:
# CELL 5: ADF Stationarity Test 

lp = df["log_price"].dropna()
adf = adfuller(lp, autolag="AIC")

print("── Augmented Dickey-Fuller Test (log copper prices) ──")
print(f"   ADF statistic : {adf[0]:.4f}")
print(f"   p-value       : {adf[1]:.4f}")
for k, v in adf[4].items():
    print(f"   Critical {k}   : {v:.4f}")

if adf[1] < 0.10:
    print("\n✅  Reject unit root → mean reversion confirmed → OU model valid")
else:
    print("\n⚠️   Cannot reject unit root at 10%. Regime-switching will handle this.")

In [ ]:
# CELL 6: OU Model Estimation (AR(1) on log prices)
def estimate_ou(df_sub, label):
    lp  = df_sub["log_price"].dropna()
    y   = lp.iloc[1:].values
    x   = lp.iloc[:-1].values
    m   = sm.OLS(y, sm.add_constant(x)).fit()
    a, b = m.params
    mu_log = a / (1 - b)
    mu_lvl = np.exp(mu_log)
    kappa  = -np.log(max(b, 1e-6))
    hl     = np.log(2) / kappa
    sigma  = m.resid.std()

    print(f"\n── OU Estimates: {label} ──")
    print(f"   β (AR coeff)    : {b:.4f}   R² = {m.rsquared:.4f}")
    print(f"   Long-run mean μ : ${mu_lvl:,.0f}/t  (~${mu_lvl/2204.6:.2f}/lb)")
    print(f"   Reversion κ     : {kappa:.4f}/month  ({kappa*12:.3f}/year)")
    print(f"   Half-life       : {hl:.1f} months  ({hl/12:.1f} years)")
    print(f"   Volatility σ    : {sigma:.4f}/month  ({sigma*np.sqrt(12):.3f}/year)")

    return dict(a=a, b=b, mu_log=mu_log, mu_lvl=mu_lvl,
                kappa=kappa, hl=hl, sigma=sigma,
                resid=m.resid, resid_idx=lp.index[1:])

# Full sample
p_full = estimate_ou(df, "Full sample 1992–2026")

# Calibration: pre-energy-transition
p_cal  = estimate_ou(df[:"2019-12-01"], "Calibration 1992–2019")



In [ ]:
# CELL 7: Chart 1 — OU Model with Bands
price  = df["price"]
s      = p_cal["sigma"]
mu_log = p_cal["mu_log"]
b1h    = np.exp(mu_log + s);   b1l = np.exp(mu_log - s)
b2h    = np.exp(mu_log + 2*s); b2l = np.exp(mu_log - 2*s)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10),
                                gridspec_kw={"height_ratios": [3, 1]})

ax1.fill_between(price.index, b2l, b2h, alpha=0.10, color=CR, label="±2σ band")
ax1.fill_between(price.index, b1l, b1h, alpha=0.20, color=CR, label="±1σ band")
ax1.axhline(p_cal["mu_lvl"],  color=CR, lw=1.8, ls="--",
            label=f"Pre-2020 OU mean: ${p_cal['mu_lvl']:,.0f}/t")
ax1.axhline(p_full["mu_lvl"], color=CB, lw=1.8, ls="-.",
            label=f"Full-sample mean: ${p_full['mu_lvl']:,.0f}/t")
ax1.plot(price.index, price.values, color="#1a1a1a", lw=1.4, label="LME Copper spot")
ax1.axvspan(pd.Timestamp("2020-01"), price.index[-1], alpha=0.06, color=CA)
ax1.axvline(pd.Timestamp("2020-01"), color=CA, lw=1, ls=":")
ax1.text(pd.Timestamp("2020-04"), b2h * 0.92, "Energy\ntransition", fontsize=8, color=CA)
ax1.set_title("LME Copper — Ornstein-Uhlenbeck Mean-Reversion Model",
              fontsize=13, fontweight="bold")
ax1.set_ylabel("USD / Metric Tonne")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.legend(fontsize=8.5, loc="upper left", framealpha=0.92)

resid_s = pd.Series(p_cal["resid"], index=p_cal["resid_idx"])
cols_r  = [CB if v > 0 else CR for v in resid_s.values]
ax2.bar(resid_s.index, resid_s.values, color=cols_r, alpha=0.65, width=20)
ax2.axhline(0, color="black", lw=0.7)
ax2.set_ylabel("Log Residual", fontsize=9)
ax2.set_title("OU Model Residuals — 1992–2019 calibration window", fontsize=9)
ax2.set_xlim(price.index[0], price.index[-1])

plt.tight_layout(pad=2)
plt.savefig("chart_01_ou_model.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊  Saved: chart_01_ou_model.png")


In [ ]:
# CELL 8: Chow Structural Break Test 
lp   = df["log_price"].dropna()
y    = lp.iloc[1:].values
x    = lp.iloc[:-1].values
idx  = lp.index[1:]
bd   = pd.Timestamp(BREAK_DATE)
pre  = idx < bd;  post = idx >= bd

def ols_ssr(y_, x_):
    m = sm.OLS(y_, sm.add_constant(x_)).fit()
    return m.ssr, len(y_), m.params

ssr_f, n_f, _       = ols_ssr(y, x)
ssr_pre, n_pre, pp  = ols_ssr(y[pre],  x[pre])
ssr_pst, n_pst, pp2 = ols_ssr(y[post], x[post])

k   = 2
F   = ((ssr_f - ssr_pre - ssr_pst) / k) / ((ssr_pre + ssr_pst) / (n_f - 2*k))
pv  = 1 - stats.f.cdf(F, k, n_f - 2*k)
mu_pre_lvl = np.exp(pp[0]  / (1 - pp[1]))
mu_pst_lvl = np.exp(pp2[0] / (1 - pp2[1]))

sig = ("✅  CONFIRMED at 1%"  if pv < 0.01 else
       "✅  CONFIRMED at 5%"  if pv < 0.05 else
       "⚠️   Marginal at 10%" if pv < 0.10 else
       "ℹ️   Not significant")

print(f"── Chow Test at {BREAK_DATE[:7]} ──────────────────────────")
print(f"   F-statistic : {F:.4f}   p-value: {pv:.4f}   {sig}")
print(f"   Pre-break long-run mean  : ${mu_pre_lvl:,.0f}/t  (~${mu_pre_lvl/2204.6:.2f}/lb)")
print(f"   Post-break long-run mean : ${mu_pst_lvl:,.0f}/t  (~${mu_pst_lvl/2204.6:.2f}/lb)")
print(f"   Implied mean shift       : +${mu_pst_lvl - mu_pre_lvl:,.0f}/t  ({(mu_pst_lvl/mu_pre_lvl-1)*100:.1f}%)")

In [ ]:
# CELL 9: Markov Regime-Switching Model

lp_s = df["log_price"].dropna()
try:
    from statsmodels.tsa.regime_switching.markov_autoregression import MarkovAutoregression
    mod = MarkovAutoregression(lp_s, k_regimes=2, order=1,
                               switching_ar=True, switching_variance=True,
                               switching_trend=True)
    best, best_llf = None, -np.inf
    np.random.seed(42)
    for _ in range(8):
        try:
            r = mod.fit(disp=False, maxiter=500)
            if r.llf > best_llf:
                best, best_llf = r, r.llf
        except Exception:
            pass

    res      = best
    smoothed = res.smoothed_marginal_probabilities

    # Identify high-mean regime
    par = res.params
    try:
        a0 = par.get("const[0]", par.iloc[0])
        a1 = par.get("const[1]", par.iloc[1])
        b0 = par.get("ar.L1[0]", par.iloc[2])
        b1 = par.get("ar.L1[1]", par.iloc[3])
        mu0 = np.exp(a0/(1-b0)) if abs(b0) < 1 else np.nan
        mu1 = np.exp(a1/(1-b1)) if abs(b1) < 1 else np.nan
    except Exception:
        mu0 = mu1 = np.nan

    high_reg  = 1 if (not np.isnan(mu1) and mu1 > mu0) else 0
    prob_high = smoothed.iloc[:, high_reg]
    ms_ok     = True
    print(f"✅  Markov model converged  LLF={res.llf:.2f}")
    if not np.isnan(mu0):
        print(f"   Low regime mean  : ${mu0:,.0f}/t")
        print(f"   High regime mean : ${mu1:,.0f}/t")

except Exception as e:
    print(f"⚠️   Markov model: {e}\n    Using rolling proxy instead.")
    roll      = lp_s.rolling(24).mean()
    mu_ov     = lp_s.mean()
    prob_high = ((roll - mu_ov) / lp_s.std()).clip(0, 1).rolling(12).mean().fillna(0)
    mu0       = np.exp(lp_s[lp_s < mu_ov].mean())
    mu1       = np.exp(lp_s[lp_s >= mu_ov].mean())
    ms_ok     = False


In [ ]:
# CELL 10: Chart 2 — Regime Detection

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10),
                                gridspec_kw={"height_ratios": [2, 1]},
                                sharex=True)

ph = prob_high.reindex(price.index).ffill().fillna(0)

# Shade high-regime periods
s_shade = None
for dt, val in ph.items():
    if val > 0.5 and s_shade is None:
        s_shade = dt
    elif val <= 0.5 and s_shade is not None:
        ax1.axvspan(s_shade, dt, alpha=0.13, color=CA)
        s_shade = None
if s_shade:
    ax1.axvspan(s_shade, price.index[-1], alpha=0.13, color=CA)

ax1.plot(price.index, price.values, color="#1a1a1a", lw=1.4, label="LME Copper spot")
ax1.axvline(pd.Timestamp(BREAK_DATE), color="red", lw=1.5, ls="--",
            label=f"Chow break: {BREAK_DATE[:7]}  F={F:.2f}  p={pv:.3f}")
if not np.isnan(mu0):
    ax1.axhline(mu0, color=CB, lw=1.2, ls=":", label=f"Low regime: ${mu0:,.0f}/t")
    ax1.axhline(mu1, color=CA, lw=1.2, ls=":", label=f"High regime: ${mu1:,.0f}/t")
ax1.set_ylabel("USD / Metric Tonne")
ax1.set_title("Copper Price Regimes — Structural Break & Markov State Detection",
              fontsize=13, fontweight="bold")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.legend(fontsize=8.5, framealpha=0.92)

ax2.fill_between(ph.index, 0, ph.values, color=CA, alpha=0.5, label="P(high-demand regime)")
ax2.plot(ph.index, ph.values, color=CA, lw=1)
ax2.axhline(0.5, color="grey", lw=0.8, ls="--")
ax2.axvline(pd.Timestamp(BREAK_DATE), color="red", lw=1.5, ls="--")
ax2.set_ylim(0, 1);  ax2.set_ylabel("P(high regime)")
ax2.set_title("Smoothed Probability of High-Demand Regime", fontsize=10)

for ds, lbl in [("2015-12","Paris\nAgreement"),("2020-07","EU Green\nDeal"),("2022-08","US IRA")]:
    dt = pd.Timestamp(ds)
    if price.index[0] <= dt <= price.index[-1]:
        ax2.axvline(dt, color="grey", lw=0.7, ls=":")
        ax2.text(dt, 0.93, lbl, ha="center", fontsize=6.5, color="grey")

ax2.legend(fontsize=9)
plt.tight_layout(pad=2)
plt.savefig("chart_02_regime.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊  Saved: chart_02_regime.png")


In [ ]:
# CELL 11: Monte Carlo Simulation

def simulate_ou(mu_lvl, kappa, sigma, S0, n_months, n_sims, seed=42):
    np.random.seed(seed)
    mu_log = np.log(mu_lvl)
    beta   = np.exp(-kappa)
    alpha  = mu_log * (1 - beta)
    paths  = np.zeros((n_months + 1, n_sims))
    paths[0] = np.log(S0)
    shocks = np.random.normal(0, sigma, (n_months, n_sims))
    for t in range(1, n_months + 1):
        paths[t] = alpha + beta * paths[t-1] + shocks[t-1]
    return np.exp(paths[1:])

def pct_bands(paths):
    return {p: np.percentile(paths, v, axis=1)
            for p, v in [("p10",10),("p25",25),("p50",50),("p75",75),("p90",90)]}

S0     = df["price"].iloc[-1]
kappa  = p_cal["kappa"]
sigma  = p_cal["sigma"]
fc_start = df.index[-1] + pd.DateOffset(months=1)
fc_idx   = pd.date_range(fc_start, periods=FC_MONTHS, freq="MS")

print(f"Starting price : ${S0:,.0f}/t")
print(f"κ={kappa:.4f}/mo   σ={sigma:.4f}/mo")
print(f"Forecast: {fc_idx[0].strftime('%Y-%m')} → {fc_idx[-1].strftime('%Y-%m')}\n")

scenarios = {}
for name, mu in [("A — Bear", MU_A), ("B — Base", MU_B), ("C — Bull", MU_C)]:
    paths = simulate_ou(mu, kappa, sigma, S0, FC_MONTHS, N_SIMS)
    scenarios[name] = pct_bands(paths)
    med = scenarios[name]["p50"][-1]
    print(f"Scenario {name}  μ=${mu:,}/t  →  Median {fc_idx[-1].year}: ${med:,.0f}/t  (~${med/2204.6:.2f}/lb)")

In [ ]:
# CELL 12: Chart 3 — Monte Carlo Fan Charts 
hist_trim = price[price.index >= price.index[-1] - pd.DateOffset(years=12)]

meta = [
    ("A — Bear", "Bear — Revert to Historical Mean",
     f"μ = ${MU_A:,}/t (~${MU_A/2204.6:.2f}/lb)  |  Supply responds; energy transition temporary",
     CR, MU_A),
    ("B — Base", "Base — Partial Regime Shift",
     f"μ = ${MU_B:,}/t (~${MU_B/2204.6:.2f}/lb)  |  Moderate structural demand; partial supply response",
     CB, MU_B),
    ("C — Bull", "Bull — Permanent High-Demand Regime",
     f"μ = ${MU_C:,}/t (~${MU_C/2204.6:.2f}/lb)  |  Structural deficit; geopolitical supply constraint",
     CG, MU_C),
]

fig, axes = plt.subplots(3, 1, figsize=(14, 16))

for ax, (sc, title, subtitle, col, mu) in zip(axes, meta):
    p = scenarios[sc]
    ax.plot(hist_trim.index, hist_trim.values, color="#1a1a1a", lw=1.8,
            label="Historical", zorder=5)
    ax.axvline(fc_idx[0], color="grey", lw=1, ls="--", alpha=0.6)
    ax.fill_between(fc_idx, p["p10"], p["p90"], color=col, alpha=0.10, label="10–90th pct")
    ax.fill_between(fc_idx, p["p25"], p["p75"], color=col, alpha=0.22, label="25–75th pct")
    ax.plot(fc_idx, p["p50"], color=col, lw=2.2, label="Median", zorder=4)
    ax.axhline(mu, color=col, lw=1, ls=":", alpha=0.7, label=f"Scenario μ: ${mu:,}/t")
    ax.axvspan(fc_idx[0], fc_idx[-1], alpha=0.03, color=col)

    end = p["p50"][-1]
    ax.annotate(f"Median {fc_idx[-1].year}:\n${end:,.0f}/t",
                xy=(fc_idx[-1], end), xytext=(-75, 20),
                textcoords="offset points", fontsize=8.5,
                color=col, fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=col, lw=1))

    ax.set_title(title, fontsize=11.5, fontweight="bold", pad=6)
    ax.text(0.005, 0.97, subtitle, transform=ax.transAxes,
            fontsize=8, color="grey", va="top")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.set_ylabel("USD / Metric Tonne", fontsize=9)
    ax.legend(fontsize=8, loc="upper left", framealpha=0.92)
    ax.set_xlim(hist_trim.index[0], fc_idx[-1])

fig.suptitle(f"Copper Price Monte Carlo Scenarios  |  {N_SIMS:,} simulations  |  OU model calibrated 1992–2019",
             fontsize=13, fontweight="bold", y=1.005)
plt.tight_layout(pad=2.5)
plt.savefig("chart_03_monte_carlo.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊  Saved: chart_03_monte_carlo.png")

In [ ]:
# CELL 13: Copper Production by Country
# Source: USGS Mineral Commodity Summaries 2026 actuals

production = {
    "Chile":       5_510,
    "DRC":         2_990,
    "Peru":        2_740,
    "China":       1_840,
    "USA":         1_050,
    "Russia":      1_020,
    "Indonesia":   1_010,
    "Australia":     765,
    "Kazakhstan":    724,
    "Mexico":        717,
    "Zambia":        823,
    "Poland":        400,
}
countries = list(production.keys())
values    = list(production.values())
colours_p = [CR if c in ("DRC","Zambia") else CA if c=="Chile" else "#AAAAAA"
             for c in countries]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(countries[::-1], values[::-1], color=colours_p[::-1], edgecolor="white")
for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=8.5)

ax.set_xlabel("Thousand Metric Tonnes — 2024 actuals")
ax.set_title("Global Copper Mine Production by Country — 2024\nSource: USGS Mineral Commodity Summaries 2026",
             fontsize=12, fontweight="bold")
ax.set_xlim(0, max(values) * 1.14)
ax.legend(handles=[
    mpatches.Patch(color=CR,       label="African producers — DRC + Zambia"),
    mpatches.Patch(color=CA,       label="Chile — dominant global supplier"),
    mpatches.Patch(color="#AAAAAA",label="Other major producers"),
], fontsize=9)

world  = 23_000
africa = production["DRC"] + production["Zambia"]
print(f"DRC + Zambia: {africa:,} kt  =  {africa/world*100:.1f}% of world supply (2024)")
print(f"COMEX average 2024: $4.22/lb  |  Projected 2025: $4.80/lb (USGS)")

plt.tight_layout()
plt.savefig("chart_04_production.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊  Saved: chart_04_production.png")


In [ ]:
#  CELL 14: Zambia Debt/GDP Scenario Analysis 
# Historical data: verified from World Bank/IMF sources
# Forecast: debt dynamics equation driven by copper price scenarios

zambia_hist = {
    2012: 23.49,
    2013: 24.22,
    2014: 44.40,   # Eurobond debt accumulation begins
    2015: 49.41,
    2016: 46.43,
    2017: 52.28,
    2018: 59.71,
    2019: 61.93,
    2020: 103.54,  # COVID + commodity crash + default (Oct 2020)
    2021: 71.41,   # GDP rebound + kwacha appreciation (denominator effect)
    2022: 56.40,   # IMF programme agreed; restructuring progress
    2023: 73.70,   # restructuring terms recognised on balance sheet
    2024: 69.60,
    2025: 69.80,
}

# Debt dynamics calibration — IMF 2024 Article IV
r_real   = 0.050    # real interest rate on debt (post-restructuring)
g_base   = 0.042    # baseline real GDP growth
pb_base  = -1.5     # baseline primary balance (% GDP)
rev_sens = 2.2      # % GDP improvement per $1,000/t copper above baseline
g_sens   = 0.003    # extra growth per $1,000/t above baseline
base_cu  = 9_300    # IMF baseline copper assumption (~$4.22/lb — 2024 actual)

def project_debt(d0, copper_path, years):
    debt = [d0]
    for i in range(1, len(years)):
        dp   = (copper_path[i-1] - base_cu) / 1_000
        pb_t = pb_base + rev_sens * dp
        g_t  = max(g_base + g_sens * dp, -0.05)
        debt.append((1 + r_real) / (1 + g_t) * debt[-1] - pb_t)
    return np.array(debt)

# Start forecast from 2025 (last actual year)
yr0      = 2025
d0       = zambia_hist[yr0]
years_fc = list(range(yr0, fc_idx[-1].year + 2))

# Get annual median copper price from Monte Carlo
debt_results = {}
for sc, col in [("A — Bear", CR), ("B — Base", CB), ("C — Bull", CG)]:
    med_mo     = pd.Series(scenarios[sc]["p50"], index=fc_idx)
    ann_median = med_mo.resample("YE").mean()
    ann_median.index = ann_median.index.year
    copper_path = [ann_median.get(y, MU_A if "Bear" in sc else MU_B if "Base" in sc else MU_C)
                   for y in years_fc[1:]]
    debt_results[sc] = (years_fc, project_debt(d0, copper_path, years_fc))

# Chart 4 — Zambia Debt
fig, ax = plt.subplots(figsize=(13, 7))

h_yrs = sorted(zambia_hist.keys())
h_vals = [zambia_hist[y] for y in h_yrs]
ax.plot(h_yrs, h_vals, color="#1a1a1a", lw=2.2, marker="o",
        markersize=4, label="Historical (verified)", zorder=5)

ax.axhline(55, color="orange", lw=1.2, ls="--", alpha=0.8, label="IMF LIC DSF threshold (55%)")
ax.axhline(70, color="red",    lw=1.2, ls="--", alpha=0.8, label="High-risk threshold (70%)")

labels_map = {"A — Bear": f"Bear (~${MU_A/2204.6:.2f}/lb)",
              "B — Base": f"Base (~${MU_B/2204.6:.2f}/lb)",
              "C — Bull": f"Bull (~${MU_C/2204.6:.2f}/lb)"}

for (sc, col) in [("A — Bear", CR), ("B — Base", CB), ("C — Bull", CG)]:
    yrs, debt = debt_results[sc]
    ax.plot(yrs, debt, color=col, lw=2.2, label=labels_map[sc])
    ax.annotate(f"{debt[-1]:.0f}%",
                xy=(yrs[-1], debt[-1]), xytext=(4, 0),
                textcoords="offset points", fontsize=9,
                color=col, fontweight="bold", va="center")

ax.axvline(yr0 + 0.5, color="grey", lw=1, ls=":", alpha=0.7)
ax.text(yr0 + 0.6, max(h_vals) * 0.98, "Forecast →", fontsize=8.5, color="grey")
ax.set_title("Zambia General Government Debt / GDP\nThree Copper Price Scenarios — 2025–2035",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Gross Debt (% of GDP)", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax.legend(fontsize=9.5, framealpha=0.95)
ax.set_xlim(2012, years_fc[-1] + 0.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("chart_05_zambia.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊  Saved: chart_05_zambia.png")


In [ ]:
# CELL 15: Full Summary

print("=" * 62)
print("MODEL SUMMARY")
print("=" * 62)

print(f"\n[OU MODEL — 1992–2019 calibration]")
print(f"  Long-run mean μ : ${p_cal['mu_lvl']:,.0f}/t  (~${p_cal['mu_lvl']/2204.6:.2f}/lb)")
print(f"  Half-life       : {p_cal['hl']:.1f} months  ({p_cal['hl']/12:.1f} years)")
print(f"  Volatility σ    : {p_cal['sigma']:.4f}/month  ({p_cal['sigma']*np.sqrt(12):.3f}/year)")

print(f"\n[CHOW TEST — break at {BREAK_DATE[:7]}]")
print(f"  F = {F:.4f}   p = {pv:.4f}   {sig}")
print(f"  Pre-break mean  : ${mu_pre_lvl:,.0f}/t")
print(f"  Post-break mean : ${mu_pst_lvl:,.0f}/t  (+{(mu_pst_lvl/mu_pre_lvl-1)*100:.1f}%)")

print(f"\n[MONTE CARLO — median price at {fc_idx[-1].strftime('%Y-%m')}]")
hdr = f"  {'Scenario':<14}  {'p10':>8}  {'p25':>8}  {'Median':>8}  {'p75':>8}  {'p90':>8}"
print(hdr);  print("  " + "─"*58)
for sc in ["A — Bear", "B — Base", "C — Bull"]:
    p = scenarios[sc]
    print(f"  {sc:<14}  "
          f"${p['p10'][-1]:>7,.0f}  ${p['p25'][-1]:>7,.0f}  "
          f"${p['p50'][-1]:>7,.0f}  ${p['p75'][-1]:>7,.0f}  ${p['p90'][-1]:>7,.0f}")

print(f"\n[ZAMBIA DEBT/GDP — 2035 projection]")
for sc in ["A — Bear", "B — Base", "C — Bull"]:
    yrs, debt = debt_results[sc]
    print(f"  {sc:<14} : {debt[-1]:.1f}%")

print(f"\n[CHARTS]")
for fn in ["chart_00_raw_prices.png","chart_01_ou_model.png","chart_02_regime.png",
           "chart_03_monte_carlo.png","chart_04_production.png","chart_05_zambia.png"]:
    print(f"  {'✅' if os.path.exists(fn) else '❌'}  {fn}")

print("\n✅  COMPLETE")

In [ ]:
# Copper Stochastic Pricing Model
**Author:** Kshitanjay Sondhi  
**Date:** April 2026  
**Data:** World Bank Commodity Markets (monthly LME copper USD/tonne, 1992–2026)  
**Required file:** Place `Copper_Prices.xlsx` in the same folder as this notebook  

**Replication:** Run Cell 1 once to install libraries, then Kernel → Restart & Run All